# Practice Lab: Context Engineering (Lesson 5.4)

Module 5 · 8 exercises · ~120-150 min
**Needs:** google-genai, tiktoken


# Lesson 5.4: Context Engineering

8 exercises: 2E + 3M + 3H
**Needs:** `pip install google-genai tiktoken`


In [ ]:
!pip install google-genai tiktoken -q


In [ ]:
from google import genai
from google.genai import types
import json, os, re, time
import numpy as np

# Gemini client + shared helpers used across all 8 exercises
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
MODEL = 'gemini-3-flash'

def count(text):
    # Token count via Gemini's tokenizer
    return client.models.count_tokens(
        model=MODEL, contents=text).total_tokens

def ask(prompt, temp=0.3):
    # Single-shot generation helper
    cfg = types.GenerateContentConfig(temperature=temp)
    return client.models.generate_content(
        model=MODEL, contents=prompt, config=cfg).text.strip()

---
## Exercise 1 (Easy): Context Stack Audit
Map the 7 layers, count tokens.


In [ ]:
# YOUR CODE
layers = {
    '1. System prompt': (
        'You are a customer support agent. '
        'Be polite. Keep responses under 100 words.'),
    '2. Few-shot examples': (
        'Customer: Order late.\nAgent: Let me check...\n' * 2),
    '3. RAG documents': 'Return policy: 7 days. ' * 10,
    '4. Tool definitions': '{"name":"check_order"}\n' * 3,
    '5. Memory': 'Customer prefers Hindi. VIP.',
    '6. History': 'Customer: Where is my order?\nAgent: Checking...\n' * 10,
    '7. User query': 'My laptop arrived damaged.',
}

# TODO: count tokens for each layer
# TODO: print budget report


<details><summary>Solution</summary>


In [ ]:
total = 0
max_ctx = 128000
print(f"Context Stack Audit (128K)\n")
print(f"{'Layer':<25} {'Tokens':>8} {'%':>6}")
print('-' * 42)
for name, content in layers.items():
    tokens = count(content)
    total += tokens
    pct = tokens / max_ctx * 100
    print(f'  {name:<23} {tokens:>6}tk '
          f'{pct:>5.1f}%')

print(f"\n  {'TOTAL':<23} {total:>6}tk "
      f"{total/max_ctx*100:>5.1f}%")
print(f"  {'REMAINING':<23} "
      f"{max_ctx-total:>6}tk "
      f"{(max_ctx-total)/max_ctx*100:>5.1f}%")


</details>


---
## Exercise 4 (Medium): Caching Architecture
Design cached prefix, calculate cost savings.


In [ ]:
# YOUR CODE
cached_prefix = {
    'system': 'You are a GST compliance expert.',
    'gst_slabs': 'GST Slabs: 0%, 5%, 12%, 18%, 28%.',
    'examples': 'Q: Laptop GST?\nA: 18% (HSN 8471).\n' * 3,
    'policies': 'Composition scheme: under 1.5 crore.',
}

# TODO: count cached tokens
# TODO: calculate cost savings at 10K calls/month


<details><summary>Solution</summary>


In [ ]:
total_cached = 0
for name, content in cached_prefix.items():
    tokens = count(content)
    total_cached += tokens
    print(f'  {name:12s}: {tokens:4d} tokens')

print(f'\n  Total cached: {total_cached} tokens')

calls = 10_000
price_per_m = 0.15
cache_discount = 0.25  # 75% off

no_cache = total_cached * calls * price_per_m / 1e6
with_cache = (total_cached * calls
              * price_per_m * cache_discount / 1e6)
savings = no_cache - with_cache

print(f'\n  No caching:  ${no_cache:.2f}/month')
print(f'  With cache:  ${with_cache:.2f}/month')
print(f'  Savings:     ${savings:.2f}/month '
      f'({savings/no_cache*100:.0f}%)')


</details>


---
## Exercise 2 (Easy): Lost-in-the-Middle Test
Hide a fact at positions 1, 5, 10, 15, 20 and check whether the model finds it (U-curve).

In [ ]:
# Exercise 2: Lost-in-the-Middle Test
cfg0 = types.GenerateContentConfig(temperature=0.0)

filler = [
    f"Paragraph {i}: India's GDP grew by "
    f"{3+i*0.2:.1f}% in Q{(i%4)+1} of 2025, "
    f"driven by manufacturing and services. "
    f"The RBI maintained rates at 6.5%. "
    f"Exports reached ${50+i}B milestone."
    for i in range(20)
]

secret = "The secret code is MANGO-42."
positions = [0, 4, 9, 14, 19]  # 1st, 5th, 10th, 15th, 20th

for pos in positions:
    paragraphs = filler.copy()
    paragraphs[pos] = secret + " " + paragraphs[pos]
    doc = "\n\n".join(paragraphs)
    prompt = (f"Read this document carefully:\n\n"
              f"{doc}\n\n"
              f"What is the secret code mentioned?")
    result = client.models.generate_content(
        model=MODEL, contents=prompt, config=cfg0).text
    found = "MANGO-42" in result
    mark = "✅" if found else "❌"
    print(f"  Position {pos+1:2d}: {mark} "
          f"{'Found' if found else 'MISSED'}")

# Expected: U-curve -- pos 1 and 20 found, pos 10 often missed

---
## Exercise 3 (Medium): Multilingual Token Audit
Compare token counts across English/Hindi/Telugu/Tamil and compute the token tax.

In [ ]:
# Exercise 3: Multilingual Token Audit
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = {
    "English": (
        "Artificial intelligence is transforming "
        "how businesses operate in India. Machine "
        "learning models can now process customer "
        "support tickets, generate reports, and "
        "analyze financial data automatically."),
    "Hindi": (
        "कृत्रिम "
        "बुद्धिमत्ता "
        "भारत में "
        "व्यापार "
        "के तरीके "
        "को बदल "
        "रही है।"),
    "Telugu": (
        "కృత్రిమ "
        "మేధస్సు "
        "భారతదేశంలో "
        "వ్యాపార "
        "విధానాలను "
        "మారుస్తుంది."),
    "Tamil": (
        "செயற்கை "
        "நுண்ணறிவு "
        "இந்தியாவில் "
        "வணிக "
        "நடைமுறைகளை "
        "மாற்றுகிறது."),
}

en_gemini = client.models.count_tokens(
    model=MODEL, contents=texts["English"]).total_tokens
en_tiktoken = len(enc.encode(texts["English"]))

print(f"{'Language':<10} {'Gemini':>8} {'tiktoken':>10} "
      f"{'Tax(G)':>8} {'Tax(T)':>8}")
print("-" * 48)
for lang, text in texts.items():
    g = client.models.count_tokens(
        model=MODEL, contents=text).total_tokens
    t = len(enc.encode(text))
    gx = g / en_gemini
    tx = t / en_tiktoken
    print(f"  {lang:<8} {g:>6}tk {t:>8}tk "
          f"{gx:>6.1f}x {tx:>6.1f}x")

print("\nBudget impact: An Indic chatbot needs "
      "2-3x more aggressive compression.")

---
## Exercise 5 (Medium): Compression Benchmark
Compress an article three ways (rule-based, extractive, abstractive); measure fact retention.

In [ ]:
# Exercise 5: Compression Benchmark
article = (
    "The OnePlus 13 launched in India at "
    "Rs 69,999 for the 12GB/256GB variant. "
    "It features a 6.82-inch 2K AMOLED display "
    "with 120Hz refresh rate, Snapdragon 8 Elite "
    "processor, 6000mAh battery with 100W "
    "charging, and a Hasselblad triple camera "
    "system (50MP main + 50MP ultrawide + 50MP "
    "telephoto with 3x zoom). It runs OxygenOS "
    "15 based on Android 15, has IP69 rating, "
    "and comes in Midnight Ocean and Arctic Dawn "
    "colors. The phone supports 5G, Wi-Fi 7, "
    "and Bluetooth 5.4. It weighs 213g. "
) * 3  # ~1000 words

questions = [
    ("What is the price?", "69,999"),
    ("What processor does it use?", "Snapdragon 8 Elite"),
    ("What is the battery size?", "6000"),
    ("What camera system?", "Hasselblad"),
    ("What is the IP rating?", "IP69"),
]

# Level 1: Rule-based compression
def compress_rules(text):
    fillers = ["it is important to note that",
               "basically", "essentially",
               "in other words", "the fact that"]
    for f in fillers:
        text = text.replace(f, "")
    return re.sub(r"\s+", " ", text).strip()

# Level 2: Extractive (API)
def compress_extract(text, n=3):
    return ask(f"Extract the {n} most important "
               f"sentences:\n{text}")

# Level 3: Abstractive (API)
def compress_abstract(text, target=50):
    return ask(f"Rewrite in ~{target} tokens. "
               f"Keep ALL facts:\n{text}")

# TODO: compress at each level
# TODO: test 5 questions on each version
# TODO: measure fact retention

---
## Exercise 6 (Hard): RAG vs Long-Context
Stuff a 20-section doc into context (Approach A); build RAG retrieval (Approach B, TODO).

In [ ]:
# Exercise 6: RAG vs Long-Context
# Simulate a long document with 20 sections
sections = []
for i in range(20):
    sections.append(
        f"Section {i+1}: {'Topic ' * 50} "
        f"Key fact {i+1}: The answer to question "
        f"{i+1} is VALUE-{i*7+3}. "
        f"{'More filler text. ' * 30}")
full_doc = "\n\n".join(sections)

questions = [
    (f"What is the answer to question {i+1}?",
     f"VALUE-{i*7+3}")
    for i in [0, 5, 10, 15, 19]
]

# Approach A: Full context stuffing
print("=== Approach A: Full Context ===")
for q, expected in questions:
    t0 = time.time()
    result = ask(f"Document:\n{full_doc}\n\n{q}")
    lat = time.time() - t0
    found = expected in result
    print(f"  {'Y' if found else 'N'} {q[:40]}... "
          f"({lat:.1f}s)")

# Approach B: RAG (embed + retrieve)
# TODO: chunk document into 500-token pieces
# TODO: embed all chunks
# TODO: for each question, retrieve top-5
# TODO: compare accuracy, latency, cost

---
## Exercise 7 (Hard): MCP Tool Context Budgeting
Count 10 tool schemas, then route to only the relevant 2-3 tools per query.

In [ ]:
# Exercise 7: MCP Tool Context Budgeting
tools = [
    {"name": "check_order", "desc": "Look up order status",
     "params": {"order_id": "string"}},
    {"name": "create_ticket", "desc": "Create support ticket",
     "params": {"subject": "str", "priority": "str"}},
    {"name": "search_products", "desc": "Search product catalog",
     "params": {"query": "str", "category": "str"}},
    {"name": "get_invoice", "desc": "Get invoice PDF",
     "params": {"invoice_id": "string"}},
    {"name": "calculate_gst", "desc": "Calculate GST",
     "params": {"amount": "float", "rate": "float"}},
    {"name": "check_inventory", "desc": "Check stock levels",
     "params": {"product_id": "str", "warehouse": "str"}},
    {"name": "schedule_delivery", "desc": "Schedule delivery",
     "params": {"order_id": "str", "date": "str"}},
    {"name": "process_refund", "desc": "Process a refund",
     "params": {"order_id": "str", "reason": "str"}},
    {"name": "send_notification", "desc": "Send SMS/email",
     "params": {"user_id": "str", "message": "str"}},
    {"name": "generate_report", "desc": "Generate sales report",
     "params": {"period": "str", "format": "str"}},
]

all_tools_text = json.dumps(tools, indent=2)
all_tokens = count(all_tools_text)
print(f"All 10 tools: {all_tokens} tokens")

# Tool router: classify and load relevant only
def route_tools(query):
    """Return only relevant tool schemas."""
    prompt = (f"Which 2-3 tools are needed for: "
              f"'{query}'?\n"
              f"Tools: {[t['name'] for t in tools]}\n"
              f"Reply with tool names only, comma-sep.")
    result = ask(prompt, temp=0.0)
    needed = [n.strip() for n in result.split(",")]
    return [t for t in tools if t["name"] in needed]

# TODO: test with 5 queries
# TODO: measure token savings per query

---
## Exercise 8 (Challenge): Production Context Pipeline
Build a ContextPipeline that budgets, trims, and summarizes a 50-turn chat.

In [ ]:
# Exercise 8: Production Context Pipeline
class ContextPipeline:
    def __init__(self, max_tokens=128000):
        self.max_tokens = max_tokens
        self.history = []
        self.budget = {
            "system": int(max_tokens * 0.05),
            "history": int(max_tokens * 0.25),
            "context": int(max_tokens * 0.30),
            "query": int(max_tokens * 0.05),
            "response": int(max_tokens * 0.35),
        }

    def add_turn(self, user_msg, assistant_msg):
        self.history.append(
            {"role": "user", "content": user_msg})
        self.history.append(
            {"role": "assistant", "content": assistant_msg})

    def trim_if_needed(self):
        """Summarize old history if over budget."""
        total = sum(count(m["content"])
                    for m in self.history)
        if total > self.budget["history"]:
            # Summarize older messages
            # Keep last 6 messages
            # TODO: implement summarize + keep recent
            pass

    def report(self, turn_num):
        total = sum(count(m["content"])
                    for m in self.history)
        pct = total / self.budget["history"] * 100
        print(f"  Turn {turn_num:2d}: "
              f"{total:>5d}tk history "
              f"({pct:.0f}% of budget)")

# TODO: simulate 50 turns
# TODO: report at turns 10, 20, 30, 40, 50
# TODO: implement trimming when budget exceeded

<details><summary>Hint - Summarize-and-forget</summary>

```python
def trim_if_needed(self):
    total = sum(count(m["content"])
                for m in self.history)
    if total <= self.budget["history"]:
        return  # within budget

    # Keep last 6 messages in full
    recent = self.history[-6:]
    old = self.history[:-6]

    # Summarize old messages
    old_text = "\n".join(
        f"{m['role']}: {m['content']}"
        for m in old)
    summary = ask(
        f"Summarize in 3 bullet points:\n"
        f"{old_text}")

    self.history = [
        {"role": "system",
         "content": f"[Summary] {summary}"}
    ] + recent
```
</details>